In [1]:
import pandas as pd
import re
from pprint import pprint
import numpy as np
cgg_path = r'data/cgg_clean_lat.tsv'
cgg_df_orig = pd.read_csv(cgg_path, sep='\t', dtype=str)

# Cleaning lonitude

In [2]:
cgg_df = cgg_df_orig.copy()

Standard cleaning


In [3]:
cgg_df['cleaned_lon'] = (cgg_df.Lon
                        .map(lambda x: re.sub(r'\s+', ' ', x), na_action='ignore')  # Removes any consecutive whitespaces
                        .map(lambda x: x.replace(',', '.'), na_action='ignore') # Standardize decimal point
                        .map(lambda x: x.replace(u'\xa0', u' '), na_action='ignore')  # Fix  weird unicode error
                        )  

Identify unique formats

In [4]:
lon_formats = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lon_formats.unique().tolist()

['-12.12',
 nan,
 '+12.12',
 '-12',
 '12.12',
 '12 12.12 E',
 "'+12.12",
 '12°12′W',
 'E12',
 '12 12 12.12 W',
 "N12 12' 12.12''",
 '12.12 E',
 '12deg12\'12.12"E',
 "E 12º12.12'",
 'E12.12',
 'E 12.12',
 'W12.12',
 '12.12W',
 "12°12'W",
 "12°12'12''E",
 '12',
 '12.12°',
 "12°12'12''",
 '12°12\'12"',
 '-12.12°W',
 '-12.12˚W',
 '12˚12\'12.12" W',
 "12 12' 12'' E",
 'W12° 12\' 12.12"',
 '12.12˚W',
 '12.12˚E',
 '12.12° E',
 '12.12 W',
 '12°12.12',
 '12°12\'12.12"E',
 '-12.12°',
 '12o12.12',
 '12° 12\' 12.12" W',
 '12.12°W',
 '-12.12˚E',
 '12°12\'12.12"',
 "12° 12' 12.12'' W",
 "12° 12'12.12'' W",
 "N12° 12' 12.12'' W",
 '12°12\'12.12"W',
 '12°12\'12.12"Ø',
 '12.12°E',
 '12°12\'12"W',
 "12° 12' 12.12'' V",
 '12°12\'12.12"V',
 '12°12.12`W',
 '12° 12’ 12.12”Ø.',
 '12° 12’ 12.12”Ø',
 '12°12’12.12’’ E',
 '12° 12 Ø',
 '12°12‘12‘‘ W',
 '12\' 12.12"',
 "12' 12.12''",
 '12° 12’ 12.12”',
 '12°12\'12" W',
 '12° 12´12´´ W',
 '12°12.12’W',
 '12° 12′ 12″ E']

### Convert to standard characters and symbols
The bad characters were found from the above step

In [5]:
cgg_df['cleaned_lon'] = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r"''|”|’’|‘‘|´´|″", '"', x), na_action='ignore')  # Replaces bad second chars with "
 .map(lambda x: re.sub(r"′|’|´|‘|`", "'", x), na_action='ignore')  # Replaces bad minute chars with '
 .map(lambda x: re.sub(r"(deg)|º|˚|o", "°", x), na_action='ignore')  # Replaces bad degree chars with °
 )

Put direction indicators (N, S, -, +) in a seperate column

In [6]:
pd.set_option('future.no_silent_downcasting', True)

cgg_df['lon_direction'] = (cgg_df.cleaned_lon
                           .map(lambda x: re.sub(r"[^A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                           .replace(np.nan, '')              
                           )
#  Removes the direction from the cleaned_lon column, as now it is no longer needed
cgg_df.cleaned_lon = (cgg_df.cleaned_lon
                      .map(lambda x: re.sub(r"[A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                      .str.strip())

Identify unique formats again

In [7]:
from pprint import pprint
lon_formats = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lon_formats = pd.Series(lon_formats.dropna().unique())
print('With degree symbol')
pprint(lon_formats.dropna()[lon_formats.dropna().str.contains('°')].unique().tolist())
print('')
print('without degree symbol')
pprint(lon_formats.dropna()[~lon_formats.dropna().str.contains('°')].unique().tolist())

With degree symbol
["12°12'",
 '12°12\'12.12"',
 "12°12.12'",
 '12°12\'12"',
 '12.12°',
 '12° 12\' 12.12"',
 '12°12.12',
 '12° 12\'12.12"',
 '12° 12\' 12.12".',
 '12° 12',
 '12° 12\'12"',
 '12° 12\' 12"']

without degree symbol
['12.12',
 '12',
 '12 12.12',
 "'12.12",
 '12 12 12.12',
 '12 12\' 12.12"',
 '12 12\' 12"',
 '12\' 12.12"']


From the above it seems safe to assume that if:
1. A coordinate contains 1 number with or without traling ° that the format is decimal degrees.
2. A coordinate contains 2 numbers seperated with with either a whitespace or ° or both that its formatted as degrees and minutes.
3. A coordinate contains 3 numbers, where first seperator is by ° or whitespace or both and second seperator is ' or whitespace or both, it is formatted as degree minute seconds.

We can formulate this using the following regular expressions:

In [8]:
dd_regex = r'''^\d{1,3}(\.\d+)?(°| °)?$'''
dm_regex = r'''^\d{1,3}( |° |°)\d{1,2}(\.\d+)?('| ')?$'''
dms_regex = r'''^\d{1,3}( |° |°)\d{1,2}( |' |')\d{1,2}(\.\d+)?("| ")?$'''

Checking the general formats 

In [9]:
def check_format(s: str):
    if re.match(dd_regex, s):
        return 'DD'
    elif re.match(dm_regex, s):
        return 'DM'
    elif re.match(dms_regex, s):
        return 'DMS'
    else:
        return 'invalid format'

classified_formats = lon_formats.apply(check_format)

lon_with_formats = pd.DataFrame({'lon': lon_formats, 'Format': classified_formats}).sort_values(by='Format')
lon_with_formats                           

,lon,Format
0,12.12,DD
1,12,DD
10,12.12°,DD
2,12 12.12,DM
16,12° 12,DM
4,12°12',DM
13,12°12.12,DM
8,12°12.12',DM
14,"12° 12'12.12""",DMS
12,"12° 12' 12.12""",DMS


Apply the same format check to the actual data

In [10]:
cgg_df['lon_format'] = cgg_df.cleaned_lon.map(check_format, na_action='ignore')

Manual check that the lon format identification was done correctly

In [11]:
for ele in cgg_df['lon_format'].unique():
    print(ele)
    lon_formats = (cgg_df.query(f"lon_format == '{ele}'").cleaned_lon
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lon_formats = pd.Series(lon_formats.dropna().unique())
    print(lon_formats)
    print()

DD
0      dd.dd
1     ddd.dd
2       d.dd
3     dd.dd°
4    ddd.dd°
5      d.dd°
dtype: object

nan
Series([], dtype: object)

invalid format
0             dddddd
1              'd.dd
2           dddddddd
3            ddddddd
4         dd°dd'ddd"
5    dd° dd' dd.dd".
6         dd° dddddd
7         dd' dd.dd"
dtype: object

DM
0      dd dd.dd
1        dd°dd'
2     dd°dd.dd'
3       ddd°dd'
4     ddd°dd.dd
5    ddd°dd.dd'
dtype: object

DMS
0         dd dd dd.dd
1      ddd dd' dd.dd"
2        ddd dd dd.dd
3       ddd°dd'dd.dd"
4           dd°dd'dd"
5        dd°dd'dd.dd"
6          dd dd' dd"
7      dd° dd' dd.dd"
8         d°dd'dd.dd"
9       d° dd' dd.dd"
10      dd° dd' d.dd"
11       dd° dd'd.dd"
12           dd°dd'd"
13    ddd° dd' dd.dd"
14         dd° dd'dd"
15        dd° dd' dd"
dtype: object



### Parse

Remove trailing non-numbers to make parsing easier

In [12]:
cgg_df.cleaned_lon = cgg_df.cleaned_lon.map(lambda x: re.sub(r"\D+$", '', x), na_action='ignore')  # removes traling non numbers

##### Split lon data into degree minute and decimal second

If it looks good apply the splitting to the data

In [13]:
cgg_df['cleaned_lon_split'] = cgg_df.cleaned_lon.str.split(pat=r"[ °']+", regex=True)

Manually inspect the splitting

In [14]:
for ele in cgg_df['lon_format'].unique():
    print(ele)
    lon_formats = (cgg_df.query(f"lon_format == '{ele}'").cleaned_lon
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lon_formats = pd.Series(lon_formats.dropna().unique())
    
    dms_formats_split = lon_formats.str.split(pat=r"[ °']+", regex=True)

    for raw, splits in zip(lon_formats, dms_formats_split):
        print(raw)
        print(splits)
        
        if ele == 'DD':
            assert len(splits) == 1
        if ele == 'DM':
            assert len(splits) == 2
        if ele == 'DMS':
            assert len(splits) == 3
    print()

DD
dd.dd
['dd.dd']
ddd.dd
['ddd.dd']
d.dd
['d.dd']

nan

invalid format
dddddd
['dddddd']
'd.dd
['', 'd.dd']
dddddddd
['dddddddd']
ddddddd
['ddddddd']
dd°dd'ddd
['dd', 'dd', 'ddd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
dd° dddddd
['dd', 'dddddd']
dd' dd.dd
['dd', 'dd.dd']

DM
dd dd.dd
['dd', 'dd.dd']
dd°dd
['dd', 'dd']
dd°dd.dd
['dd', 'dd.dd']
ddd°dd
['ddd', 'dd']
ddd°dd.dd
['ddd', 'dd.dd']

DMS
dd dd dd.dd
['dd', 'dd', 'dd.dd']
ddd dd' dd.dd
['ddd', 'dd', 'dd.dd']
ddd dd dd.dd
['ddd', 'dd', 'dd.dd']
ddd°dd'dd.dd
['ddd', 'dd', 'dd.dd']
dd°dd'dd
['dd', 'dd', 'dd']
dd°dd'dd.dd
['dd', 'dd', 'dd.dd']
dd dd' dd
['dd', 'dd', 'dd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
d°dd'dd.dd
['d', 'dd', 'dd.dd']
d° dd' dd.dd
['d', 'dd', 'dd.dd']
dd° dd' d.dd
['dd', 'dd', 'd.dd']
dd° dd'd.dd
['dd', 'dd', 'd.dd']
dd°dd'd
['dd', 'dd', 'd']
ddd° dd' dd.dd
['ddd', 'dd', 'dd.dd']
dd° dd'dd
['dd', 'dd', 'dd']
dd° dd' dd
['dd', 'dd', 'dd']



Test if the different formats have correct number of elementes in the split lists and that each element only contains numbers

In [15]:
def all_numbers(lst):
    return all(s.replace(".", "", 1).isdigit() for s in lst)

for i, row in cgg_df[['lon_format', 'cleaned_lon_split']].iterrows():
    split_lst = row.cleaned_lon_split
    lon_format = row.lon_format
    
    if lon_format == 'DD':
        assert len(split_lst) == 1
    if lon_format == 'DM':
        assert len(split_lst) == 2
    if lon_format == 'DMS':
        assert len(split_lst) == 3
        
    if isinstance(split_lst, list) and lon_format != 'invalid format':
        for ele in split_lst:
            try:
                float(ele)
            except ValueError:
                raise Exception(f'bad list: {split_lst}')

Convert direction to plus and minus

In [16]:
def convert_direction(row):
    direction = str(row.lon_direction)
    if direction == 'nan':
        return np.nan
    direction = re.sub(r'[EeØø]', '+', direction)
    direction = re.sub(r'[WwVv]', '-', direction)
    direction = direction.strip()
    if bool(re.match(r'^(\-|\+)$', direction)) or direction == '':
        return direction
    else:
        return 'invalid direction'

cgg_df['converted_lon_direction'] = cgg_df.apply(convert_direction, axis=1)

Convert Split DM data into decimal degrees and add converted direction

In [17]:
def add_direction(row):
    direction = str(row.converted_lon_direction)
    coord = row.converted_lon
    if not direction == 'invalid direction':
        return float(str(direction) + str(coord))
    else:
        return coord

def convert_dd(lst):
    assert len(lst) == 1
    return float(lst[0])

def convert_dm(lst):
    assert len(lst) == 2
    degrees, minutes = float(lst[0]), float(lst[1])
    
    return degrees + (minutes/60)

def convert_dms(lst):
    assert len(lst) == 3
    degrees, minutes, seconds = float(lst[0]), float(lst[1]), float(lst[2])
    
    return degrees + (minutes/60) + (seconds/3600)

def convert_to_dd(row):
  
    lon_format = row.lon_format 
    split_lst = row.cleaned_lon_split
    
    if lon_format == 'DD':
        result = convert_dd(split_lst)
    elif lon_format == 'DM':
        result = convert_dm(split_lst)
    elif lon_format == 'DMS':
        result = convert_dms(split_lst)
    else:
        return np.nan
    return result
    
cgg_df['converted_lon'] = cgg_df.apply(convert_to_dd, axis=1)
cgg_df['converted_lon'] = cgg_df.apply(add_direction, axis=1)

Check that the conversion was done correctly

In [18]:
lon_formats = (cgg_df.converted_lon
 .map(lambda x: re.sub(r'\d', 'd', str(x)), na_action='ignore')  # Turns all n sized numbers into 123
 )
lon_formats.unique().tolist()

['-dd.dddd',
 nan,
 '-ddd.ddddd',
 'ddd.ddddd',
 'd.dddddd',
 'dd.dddd',
 'ddd.dddd',
 'ddd.dddddd',
 'dd.ddddd',
 'dd.dddddd',
 'ddd.ddd',
 'd.dddd',
 'd.ddd',
 'dd.ddd',
 '-dd.ddddddddddddddd',
 'ddd.dd',
 '-dd.dddddddddddddd',
 '-ddd.dddddd',
 'dd.dddddddddddddd',
 '-ddd.ddddddddddddd',
 'ddd.dddddddddddddd',
 'd.ddddd',
 '-dd.ddd',
 '-ddd.dd',
 'd.ddddddddddddddd',
 'd.dddddddddddddd',
 'dd.ddddddddddddddd',
 '-dd.ddddd',
 '-d.dddddd',
 '-d.ddddd',
 'dd.d',
 'dd.dddddddddddd',
 'dd.ddddddddddddd',
 '-d.dd',
 'dd.dd',
 'd.dd',
 '-ddd.ddd',
 'ddd.ddddddd',
 '-ddd.dddddddd',
 '-dd.dddddd',
 '-d.dddd',
 '-dd.ddddddddddddd',
 '-dd.ddddddd',
 '-ddd.ddddddd',
 'd.ddddddd',
 'dd.dddddddd',
 '-dd.dddddddd',
 '-d.ddddddddddddddd',
 '-dd.dd',
 'd.ddddddddd',
 '-ddd.dddd',
 '-d.ddddddd']

In [19]:
cgg_df.query('converted_lon_direction == "invalid direction"')['lon_direction'].unique()

array(['N', '-W', '-E', 'NW'], dtype=object)

Manually inspect the data to verify

In [20]:
cgg_df.query('lon_format == "DMS"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction', 'converted_lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction,converted_lon_direction
15533,"12°35'53.719""",12°35'53.719,12.598255,DMS,,
7551,"49˚54'57.80"" W",49°54'57.80,-49.916056,DMS,W,-
10700,"8°27'57.88""E",8°27'57.88,8.466078,DMS,E,+
17544,"13°58'0"" W",13°58'0,-13.966667,DMS,W,-
11939,"1° 23' 17.232"" W",1° 23' 17.232,-1.388120,DMS,W,-
12370,N20° 22' 59.5848'' W,20° 22' 59.5848,20.383218,DMS,NW,invalid direction
10463,"46°53'29.17""E",46°53'29.17,46.891436,DMS,E,+
12658,"19°10'13.0""W",19°10'13.0,-19.170278,DMS,W,-
11941,"1° 23' 17.232"" W",1° 23' 17.232,-1.388120,DMS,W,-
2567,N056 20' 15.66'',056 20' 15.66,56.337683,DMS,N,invalid direction


In [21]:
cgg_df.query('lon_format == "DM"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
10363,015°11.391,015°11.391,15.18985,DM,
5021,113°27'W,113°27,-113.45000,DM,W
11285,015o04.632,015°04.632,15.07720,DM,
5056,113°27'W,113°27,-113.45000,DM,W
5010,113°27'W,113°27,-113.45000,DM,W
13276,071°18.180`W,071°18.180,-71.30300,DM,W
5098,113°27'W,113°27,-113.45000,DM,W
5018,113°27'W,113°27,-113.45000,DM,W
10320,015°11.391,015°11.391,15.18985,DM,
3671,E 93º24.429',93°24.429,93.40715,DM,E


In [22]:
cgg_df.query('lon_format == "DD"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
1714,+9.3523,9.3523,9.352300,DD,+
7906,24.932,24.932,24.932000,DD,
16819,"74,75W",74.75,-74.750000,DD,W
8433,50.19411˚W,50.19411,-50.194110,DD,W
19639,"-0,940515",0.940515,-0.940515,DD,-
8069,-5.24,5.24,-5.240000,DD,-
9931,172.619669,172.619669,172.619669,DD,
14766,120.169786,120.169786,120.169786,DD,
19358,"0,51645",0.51645,0.516450,DD,
14192,120.245163,120.245163,120.245163,DD,


In [23]:
cgg_df.query('lon_format == "invalid format"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
20845,"29' 32.522""",29' 32.522,NaN,invalid format,
20575,"29' 32.522""",29' 32.522,NaN,invalid format,
16263,"28' 15.24""",28' 15.24,NaN,invalid format,
16343,29' 32.49'',29' 32.49,NaN,invalid format,
5397,51848361,51848361,NaN,invalid format,
16271,"28' 15.24""",28' 15.24,NaN,invalid format,
5387,51829806,51829806,NaN,invalid format,
16935,19' 29.70'',19' 29.70,NaN,invalid format,
15564,11° 100158 Ø,11° 100158,NaN,invalid format,Ø
20596,19' 29.70'',19' 29.70,NaN,invalid format,


In [24]:
cgg_df.to_csv(r'data/cgg_clean_lat_lon.tsv', sep='\t', index=False)

### Clean invalid formatted coordinates

Check coords where direction = invalid direction

Check if any lon directions are bad

Mark rows that contain E, W, e or w in lon as invalid, bad_direction

In [25]:
invalid_filter = cgg_df.lon_direction.str.contains(r'E|W|e|w', na=False)

In [26]:
cgg_df[invalid_filter]['invalid_format'] = True
cgg_df[invalid_filter]['invalid_format_comment'] = 'bad_direction'

C:\Users\glj523\AppData\Local\Temp\ipykernel_10472\311128983.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cgg_df[invalid_filter]['invalid_format'] = True
C:\Users\glj523\AppData\Local\Temp\ipykernel_10472\311128983.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cgg_df[invalid_filter]['invalid_format_comment'] = 'bad_direction'


Do format checking on invalid ones, replace digits with d

lons that are probably missing a decimal

In [27]:
no_dec_filter = cgg_df.cleaned_lon.map(lambda x: bool(re.match(pattern=r'^\d+$', string=x)), na_action='ignore')
sorted(cgg_df[no_dec_filter].cleaned_lon.unique())

ValueError: Cannot mask with non-boolean array containing NA / NaN values